In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ── MDP & algorithm components ───────────────────────────────────────────────
from principal_agent_mdp import PrincipalAgentMDP

# oracle (meta / value-iteration)
from meta.agent_meta import AgentMeta
from meta.principal_meta import PrincipalMeta

# deep Q-learning
from principal_dq import PrincipalDQ
from agent_dq import AgentDQ
from deepq_network import QNetwork, ReplayBuffer

# ── device ───────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
"""
compare_deepql_vs_oracle.py
────────────────────────────
Compares deep Q-learning (Algorithm 3) against the exact meta / value-iteration
oracle across N_MDP randomly generated principal-agent MDPs.

Drop-in replacement for the Q-learning comparison script:
  • solve_exact  — unchanged (meta value-iteration oracle)
  • solve_deepql — replaces solve_qlearning; re-initialises all networks per MDP
  • evaluate_utility — unchanged
  • plotting     — unchanged
"""


# ── shared run parameters ────────────────────────────────────────────────────
N_MDP      = 100       # number of random MDPs to evaluate
GAMMA      = 1.0
N_META     = 20        # oracle iterations

# deep Q hyper-parameters (as for the normal iterations)
NUM_ITERATIONS  = 20_000
NUM_INTERACTIONS = 8
BATCH_SIZE      = 128
WARMUP          = 1_000
BUFFER_CAPACITY = 20_000
HIDDEN_DIM      = 256
LR_INIT         = 1e-3
LR_FINAL        = 1e-4
TARGET_UPDATE   = 100   # hard-copy every N iterations
EPSILON_START   = 1.0
EPSILON_END     = 0.0


# ── helpers ──────────────────────────────────────────────────────────────────
def to_one_hot(s: int, n_states: int) -> torch.Tensor:
    v = torch.zeros(n_states, dtype=torch.float32, device=device)
    v[s] = 1.0
    return v


def parameter_update(source, target, tau: float = 1.0):
    """Soft (or hard when tau=1) update: target ← tau*source + (1-tau)*target."""
    for t_p, s_p in zip(target.parameters(), source.parameters()):
        t_p.data.copy_(tau * s_p.data + (1.0 - tau) * t_p.data)


def make_networks(n_states: int, n_actions: int):
    """Fresh copies of both Q-networks (online + target) for one MDP."""
    q_theta       = QNetwork(n_states, n_actions, HIDDEN_DIM).to(device)
    q_theta_prime = QNetwork(n_states, n_actions, HIDDEN_DIM).to(device)
    Q_phi         = QNetwork(n_states, n_actions, HIDDEN_DIM).to(device)
    Q_phi_prime   = QNetwork(n_states, n_actions, HIDDEN_DIM).to(device)
    q_theta_prime.load_state_dict(q_theta.state_dict())
    Q_phi_prime.load_state_dict(Q_phi.state_dict())
    return q_theta, q_theta_prime, Q_phi, Q_phi_prime


# ── oracle ───────────────────────────────────────────────────────────────────
def solve_exact(mdp, r_p):
    """Meta value-iteration oracle — identical to the Q-learning comparison."""
    agent     = AgentMeta(mdp)
    principal = PrincipalMeta(mdp, r_p, b_grid_step=0.1)
    rho       = {s: np.zeros(2) for s in range(mdp.n_states)}

    for _ in range(N_META):
        agent.solve(rho)
        principal.solve(agent)
        rho = principal.rho_star

    return principal.V[mdp.s0], rho


# ── deep Q-learning ──────────────────────────────────────────────────────────
def solve_deepql(mdp, r_p):
    """
    One run of Algorithm 2 for a single MDP.

    All networks and optimisers are created fresh so that nothing leaks
    between MDPs when called inside the N_MDP loop.
    """
    principal = PrincipalDQ(mdp, r_p, epsilon=EPSILON_START)
    agent     = AgentDQ(mdp, epsilon=EPSILON_START)

    q_theta, q_theta_prime, Q_phi, Q_phi_prime = make_networks(
        mdp.n_states, mdp.n_actions
    )

    opt_theta = torch.optim.Adam(q_theta.parameters(), lr=LR_INIT)
    opt_phi   = torch.optim.Adam(Q_phi.parameters(),   lr=LR_INIT)

    lr_decay = (LR_FINAL / LR_INIT) ** (1.0 / NUM_ITERATIONS)
    sched_theta = torch.optim.lr_scheduler.ExponentialLR(opt_theta, gamma=lr_decay)
    sched_phi   = torch.optim.lr_scheduler.ExponentialLR(opt_phi,   gamma=lr_decay)

    buffer = ReplayBuffer(BUFFER_CAPACITY)

    # contract initialised to zero payment for all outcomes
    rho = {s: (0.0, 0.0) for s in range(mdp.n_states)}

    s = mdp.s0

    for num in range(NUM_ITERATIONS):
        # ── linear epsilon annealing ─────────────────────────────────────────
        eps = EPSILON_START - (EPSILON_START - EPSILON_END) * (num / NUM_ITERATIONS)
        principal.epsilon = eps
        agent.epsilon     = eps

        # ── environment interactions ─────────────────────────────────────────
        for _ in range(NUM_INTERACTIONS):
            a_p    = principal.induce_action(s, q_theta)
            o      = mdp.sample_outcome(s, a_p)
            s_next = mdp.T(s, o)
            r_a    = mdp.R_agent(s, a_p, rho[s], o)
            rwrd_p = principal.r_p[s, o]
            done   = int(mdp.is_terminal(s_next))

            buffer.append((s, a_p, r_a, rwrd_p, o, done, s_next))

            s = mdp.s0 if done else s_next

        if len(buffer) < WARMUP:
            continue

        # ── sample mini-batch ────────────────────────────────────────────────
        states, actions, r_agent, r_principal, outcomes, dones, next_states = (
            buffer.sample(BATCH_SIZE)
        )

        states_oh      = F.one_hot(states.long(),      num_classes=mdp.n_states).float().to(device)
        next_states_oh = F.one_hot(next_states.long(), num_classes=mdp.n_states).float().to(device)

        # next action from online principal network
        a_p_prime = q_theta(next_states_oh).detach().argmax(dim=1)

        # ── compute optimal contracts for the batch ──────────────────────────
        b_star       = []
        b_star_prime = []
        for idx in range(BATCH_SIZE):
            s_i      = int(states[idx].item())
            s_i_next = int(next_states[idx].item())
            a_i      = int(actions[idx].item())
            a_i_p    = int(a_p_prime[idx].item())

            Q_bar_s      = Q_phi(to_one_hot(s_i,      mdp.n_states)).detach().cpu().numpy()
            Q_bar_s_next = Q_phi(to_one_hot(s_i_next, mdp.n_states)).detach().cpu().numpy()

            b_star.append(principal.find_best_contract(s_i,      a_i,   Q_bar_s))
            b_star_prime.append(principal.find_best_contract(s_i_next, a_i_p, Q_bar_s_next))

        # outcome-indexed payment b*(o) for each transition
        b_star_o = torch.tensor(
            [b_star[i][int(outcomes[i].item())] for i in range(BATCH_SIZE)],
            dtype=torch.float32, device=device
        )

        # ── principal target y_p ─────────────────────────────────────────────
        q_target_vals = (
            q_theta_prime(next_states_oh)
            .gather(1, a_p_prime.view(-1, 1))
            .squeeze()
            .detach()
        )
        y_p = r_principal - b_star_o + (1 - dones.float()) * q_target_vals

        # ── agent target y_a ─────────────────────────────────────────────────
        E_b_prime = torch.tensor(
            [agent._expected_payment(
                int(next_states[i].item()),
                int(a_p_prime[i].item()),
                b_star_prime[i])
             for i in range(BATCH_SIZE)],
            dtype=torch.float32, device=device
        )
        Q_phi_prime_vals = (
            Q_phi_prime(next_states_oh)
            .gather(1, a_p_prime.view(-1, 1))
            .squeeze()
            .detach()
        )
        y_a = r_agent + (1 - dones.float()) * (E_b_prime + Q_phi_prime_vals)

        # ── principal network update L(θ) ────────────────────────────────────
        q_pred_theta = q_theta(states_oh).gather(1, actions.view(-1, 1)).squeeze()
        loss_theta   = F.mse_loss(q_pred_theta, y_p)
        opt_theta.zero_grad()
        loss_theta.backward()
        opt_theta.step()
        sched_theta.step()

        # ── agent network update L(φ) ────────────────────────────────────────
        q_pred_phi = Q_phi(states_oh).gather(1, actions.view(-1, 1)).squeeze()
        loss_phi   = F.mse_loss(q_pred_phi, y_a)
        opt_phi.zero_grad()
        loss_phi.backward()
        opt_phi.step()
        sched_phi.step()

        # ── hard target-network copy every TARGET_UPDATE steps ───────────────
        if num % TARGET_UPDATE == 0:
            parameter_update(q_theta, q_theta_prime, tau=1.0)
            parameter_update(Q_phi,   Q_phi_prime,   tau=1.0)

    # ── extract learned contracts ρ from the trained principal ───────────────
    for s in range(mdp.n_states):
        Q_bar_s = Q_phi(to_one_hot(s, mdp.n_states)).detach().cpu().numpy()
        best_a  = int(q_theta(to_one_hot(s, mdp.n_states)).detach().argmax().item())
        rho[s]  = principal.find_best_contract(s, best_a, Q_bar_s)

    return evaluate_utility(mdp, rho)


# ── Monte-Carlo evaluation (unchanged) ───────────────────────────────────────
def evaluate_utility(mdp, rho, n_eval: int = 1_000):
    """Monte Carlo evaluation of principal utility under ρ."""
    returns = []
    for _ in range(n_eval):
        s, vp, t = mdp.s0, 0.0, 0
        while not mdp.is_terminal(s):
            b = rho[s] if not isinstance(rho[s], tuple) else np.array(rho[s])
            Q_full = np.array([
                sum(mdp.P_outcome[s, a, o] * mdp.R_agent(s, a, b, o)
                    for o in range(mdp.n_outcomes))
                for a in range(mdp.n_actions)
            ])
            a   = int(np.argmax(Q_full))
            o   = mdp.sample_outcome(s, a)
            vp += (GAMMA ** t) * mdp.R_principal(s, b, o)
            t  += 1
            s   = mdp.T(s, o)
        returns.append(vp)
    return np.mean(returns)


# ── main loop ─────────────────────────────────────────────────────────────────
exact_utils, dql_utils, regrets = [], [], []

for i in tqdm(range(N_MDP), desc="MDPs"):
    # ── build a fresh random MDP ─────────────────────────────────────────────
    mdp     = PrincipalAgentMDP(gamma=GAMMA)
    cost_aL = np.random.uniform(0, 1)

    for s in range(mdp.n_states):
        for a in range(mdp.n_actions):
            probs = np.random.dirichlet(3 * np.ones(mdp.n_outcomes))
            mdp.P_outcome[s, a, :] = probs

    r_p = np.zeros((mdp.n_states, mdp.n_outcomes))
    for s in range(mdp.n_states):
        r_p[s, 0] = 14 / 9
        r_p[s, 1] = 0.0

    def R_agent_rand(s, a, b, o, c=cost_aL):
        return (-c if a == 0 else 0.0) + b[o]

    def R_principal_rand(s, b, o, rp=r_p):
        return rp[s, o] - b[o]

    mdp.R_agent     = R_agent_rand
    mdp.R_principal = R_principal_rand

    # ── run both algorithms ──────────────────────────────────────────────────
    u_exact, _ = solve_exact(mdp, r_p)
    u_dql      = solve_deepql(mdp, r_p)

    exact_utils.append(u_exact)
    dql_utils.append(u_dql)
    regrets.append(u_exact - u_dql)


# ── summary ───────────────────────────────────────────────────────────────────
print("\nDone!")
print(f"Exact utility  — mean: {np.mean(exact_utils):.4f}, std: {np.std(exact_utils):.4f}")
print(f"Deep-QL utility— mean: {np.mean(dql_utils):.4f},   std: {np.std(dql_utils):.4f}")
print(f"Regret         — mean: {np.mean(regrets):.4f},    std: {np.std(regrets):.4f}")
print(
    f"Regret %       — mean: "
    f"{np.mean(np.array(regrets) / (np.array(exact_utils) + 1e-9) * 100):.2f}%"
)

# ── plots (same layout as Q-learning comparison) ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(exact_utils, dql_utils, alpha=0.5)
lims = [min(exact_utils + dql_utils), max(exact_utils + dql_utils)]
axes[0].plot(lims, lims, "r--", label="perfect match")
axes[0].set_xlabel("Exact SPE utility")
axes[0].set_ylabel("Deep Q-learning utility")
axes[0].set_title("Exact vs Deep Q-learning")
axes[0].legend()

axes[1].hist(regrets, bins=20, edgecolor="black", color="orange")
axes[1].axvline(np.mean(regrets), color="red",   linestyle="--",
                label=f"mean={np.mean(regrets):.3f}")
axes[1].axvline(0,                color="green", linestyle="-", label="zero regret")
axes[1].set_title("Regret distribution")
axes[1].set_xlabel("Regret (exact − deep QL)")
axes[1].legend()

axes[2].hist(exact_utils, bins=20, alpha=0.5, label="Exact SPE",      color="blue")
axes[2].hist(dql_utils,   bins=20, alpha=0.5, label="Deep Q-learning", color="orange")
axes[2].axvline(np.mean(exact_utils), color="blue",   linestyle="--")
axes[2].axvline(np.mean(dql_utils),   color="orange", linestyle="--")
axes[2].set_title("Utility distributions")
axes[2].set_xlabel("Principal utility")
axes[2].legend()

plt.tight_layout()
plt.savefig("comparison_deepql.png", dpi=150)
plt.show()

NameError: name 'q_theta' is not defined